# Retrieval as prompting (offline)

**Session 5 · small model (`llama3.2:3b`)**

A minimal, offline RAG: local embeddings + numpy cosine + a grounded prompt. Then measure
three things separately, because "it works" hides which part works:

- **answer accuracy** on questions the docs *can* answer,
- **refusal rate** on questions they *cannot*,
- **distractor resistance**: a near-duplicate doc about a different library is in the corpus.

Finally, weaken the grounding instruction and watch the refusal rate fall — a number, not a hunch.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import numpy as np
import ollama  # needs the local Ollama app + `ollama pull nomic-embed-text`
from utils import ask, SMALL_MODEL, BIG_MODEL
from eval import load_cases

### The retriever + a grounded prompt

Embed the 9 library docs, cosine-match the top 3 to the question, and answer only from them.

In [ ]:
DOCS = [d["text"] for d in load_cases("../eval/datasets/rag_docs.jsonl")]
print(f"{len(DOCS)} docs. Note doc 1 (Central Library, 9am) vs doc 9 (University Library, 8am) -- the distractor.")

def embed(t):
    return np.array(ollama.embeddings(model="nomic-embed-text", prompt=t)["embedding"])

DOC_VECS = [embed(d) for d in DOCS]

def retrieve(q, k=3):
    qv = embed(q)
    sims = [float(qv @ v / (np.linalg.norm(qv) * np.linalg.norm(v))) for v in DOC_VECS]
    order = np.argsort(sims)[::-1][:k]
    return [DOCS[i] for i in order]

GROUNDED = ("Answer ONLY from the context. Cite the sentence you used with [n]. "
            "If the context does not contain the answer, reply exactly: I do not know.")

def answer(q, instruction=GROUNDED, model=SMALL_MODEL):
    ctx = "\n".join(f"[{i + 1}] {c}" for i, c in enumerate(retrieve(q)))
    return ask(f"{instruction}\n\n{ctx}\n\nQ: {q}", model=model)

print(answer("What time does the Central Library open on weekdays?"))   # distractor nearby
print(answer("What is the wifi password?"))                             # not in docs


### Score it: three rates, not one

Each question is tagged `answerable`, `distractor`, or `refuse`, with an `expected` string
(or `"REFUSE"`). We check the right behaviour per tag.

In [ ]:
QUESTIONS = load_cases("../eval/datasets/rag_questions.jsonl")

def refused(ans):
    a = ans.lower()
    return "do not know" in a or "don't know" in a or "cannot find" in a or "not in the" in a

def evaluate(instruction, model=SMALL_MODEL):
    buckets = {}
    for q in QUESTIONS:
        ans = answer(q["input"], instruction, model)
        if q["expected"] == "REFUSE":
            ok = refused(ans)
        else:
            ok = (not refused(ans)) and q["expected"].lower() in ans.lower()
        buckets.setdefault(q["kind"], []).append(ok)
    for kind, xs in buckets.items():
        print(f"  {kind:11} {sum(xs)}/{len(xs)}")
    return buckets

print("grounded instruction:")
_ = evaluate(GROUNDED)


### The grounding instruction is doing the work — prove it

Swap the strict instruction for a lax one and re-score. The `refuse` bucket collapses: without
the explicit "reply I do not know" clause the model fills gaps from its own memory.

In [ ]:
LAX = "Use the context below to help answer the question."

print("strict grounding:")
strict = evaluate(GROUNDED)
print("\nlax instruction:")
lax = evaluate(LAX)

def refuse_rate(b):
    xs = b.get("refuse", [])
    return sum(xs) / len(xs) if xs else 0.0

print(f"\nrefusal rate:  strict {refuse_rate(strict):.0%}   ->   lax {refuse_rate(lax):.0%}")


## Your turn - vary the example

1. `retrieve()` returns `k=3` chunks. Drop to `k=1` — does the distractor question get worse?
   Raise to `k=6` — does refusal get worse because the extra irrelevant text invites guessing?
2. Add a question whose answer is genuinely split across two docs. Does the bot combine them?
3. The `refused()` check is a keyword hack. Find an answer it misclassifies and tighten it.
4. Run `evaluate(GROUNDED, model=BIG_MODEL)`. Does the big model refuse more reliably, or does
   it also need the explicit instruction?